In [1]:
import os
import sys
import numpy as np
from pathlib import Path

from torchvision import transforms
from PIL import Image

import torch
from compressai.zoo import cheng2020_anchor

import h5py

In [2]:
# Si tenemos disponible GPU, lo usamos
# Chequeamos si tenemos disponible GPU (CUDA)
if torch.cuda.is_available():
    device = "cuda"
# Chequeamos si tenemos disponible aceleración por hardware en un chip de Apple (MPS)
elif torch.backends.mps.is_available():
    device = "mps"
# Por defecto usamos CPU
else:
    device = "cpu"

In [3]:
# Me aseguro de que el directorio raíz del proyecto esté en el sys.path
project_root = Path(os.path.abspath("")).parent

# Añado el directorio raíz al sys.path si no está ya presente
if project_root not in sys.path:
    sys.path.append(str(project_root))

In [4]:
# Importo las funciones de configuración
from src.config import interim_data_dir, raw_data_dir, models_dir, load_config

from src.utils.datasets import CustomDataset

#from src.models.convolutional_autoencoder_model.model import ConvolutionalAutoencoder
#from src.models.compressai_chang2020_model.train_model import train_model
from src.models.compressai_chang2020_model.image_processing import pad_image_to_multiple
from src.models.compressai_chang2020_model.inference import compress_image
from src.models.compressai_chang2020_model.compression_utils import save_compressed_data_to_h5, save_compressed_data_to_h5_2

Current working directory: /home/jorge/development/ImageReconstructionDL/notebooks
Loading configuration from /home/jorge/development/ImageReconstructionDL/src/config.yaml


In [5]:
# Verifico que el dispositivo que se está utilizando
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [6]:
nombre_modelo = "compressai_cheng2020_anchor_2_2_BIS.pth"
models_path = models_dir() / "trained"
model_path = models_path / nombre_modelo
# convierto a string
model_path = str(model_path)

# Cargo el modelo
modelo = cheng2020_anchor(quality=1, pretrained=True)
modelo.load_state_dict(torch.load(model_path, map_location=device))
modelo.eval().to(device)


Cheng2020Anchor(
  (entropy_bottleneck): EntropyBottleneck(
    (likelihood_lower_bound): LowerBound()
  )
  (g_a): Sequential(
    (0): ResidualBlockWithStride(
      (conv1): Conv2d(3, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (leaky_relu): LeakyReLU(negative_slope=0.01, inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (gdn): GDN(
        (beta_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
        (gamma_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
      )
      (skip): Conv2d(3, 128, kernel_size=(1, 1), stride=(2, 2))
    )
    (1): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (leaky_relu): LeakyReLU(negative_slope=0.01, inplace=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (2): ResidualBlockWithStride(
      (conv1): Conv2d(1

In [7]:
# Cargo una imagen y la convierto a tensor de 4 dimensiones
imagen_path = raw_data_dir() / "press" / "data" / "00005-1062776717.png"

imagen = Image.open(imagen_path).convert("RGB")
imagen = pad_image_to_multiple(imagen)

# Transformo la imagen a tensor
x = transforms.ToTensor()(imagen).unsqueeze(0).to(device)

In [8]:
# Inferencia y compresión
datos_comprimidos = compress_image(modelo, x)

In [9]:
save_compressed_data_to_h5(datos_comprimidos, 'comprimido_cae.h5')